## Load Dataset


In [ ]:
from google.colab import drive

# 1. Mount Drive

drive.mount('/content/drive')

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="7uTyxQ8GJz6qALNlVUlk")
project = rf.workspace("workspace1-vcchj").project("fire123_seg")
version = project.version(1)
dataset = version.download("yolo26")

In [ ]:
!pip install ultralytics

## Train

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# 2. Set up paths
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "output"
dataset_yaml_path = f"{ROOT}/fire123_seg-1/data.yaml"

# 3. Model
# model = YOLO("yolo26n-seg.pt")
model = YOLO("/content/drive/MyDrive/Models/firesmoke_3_seg/weights/best.pt")

# 4. Train
results = model.train(
    data=dataset_yaml_path,
    epochs=300,
    imgsz=640,
    batch=32,
    project=str(OUTPUT_DIR),
    name="firesmoke_4_merged_seg",
    val=True,
    seed=42,
    patience=70,
    plots=True,
    # --- FINE-TUNING
    # adamW
    lr0=0.0005,
    lrf=0.01,         # Final learning rate fraction (0.0005 * 0.01)
    # 2. Smooth LR Decay
    cos_lr=True,
    # 3. Shorter Warmup
    warmup_epochs=1.0,
    # --- AUGMENTATIONS
    # 4. Blending
    mixup=0.15,
    copy_paste=0.3,
    mosaic=1.0
)

## Generate Metrics

In [ ]:
from ultralytics import YOLO

# 1. Load model
model = YOLO('/content/output/firesmoke_4_merged_seg-3/weights/best.pt')

# 2. Run validation
metrics = model.val(data='/content/fire123_seg-1/data.yaml')

print("Validation complete! Check your /content/runs/segment/val folder for the plots.")

## Fix dataset

In [ ]:
import os
from pathlib import Path

# Set this to your dataset directory
DATASET_DIR = Path("/content/fire123_seg-1")

def fix_bbox_to_polygon(label_path):
    fixed_count = 0
    with open(label_path, 'r') as f:
        lines = f.readlines()

    new_lines = []
    for line in lines:
        parts = line.strip().split()

        # If a line has exactly 5 values, it's a bounding box, not a polygon mask
        if len(parts) == 5:
            cls, xc, yc, w, h = map(float, parts)
            cls = int(cls)

            # Convert center x,y and width,height to 4 corners
            x1, y1 = xc - w/2, yc - h/2 # Top-left
            x2, y2 = xc + w/2, yc - h/2 # Top-right
            x3, y3 = xc + w/2, yc + h/2 # Bottom-right
            x4, y4 = xc - w/2, yc + h/2 # Bottom-left

            # Ensure coordinates stay within 0.0 to 1.0 bounds
            coords = [x1, y1, x2, y2, x3, y3, x4, y4]
            coords = [max(0.0, min(1.0, c)) for c in coords]

            # Format as a polygon string
            new_line = f"{cls} " + " ".join([f"{c:.6f}" for c in coords]) + "\n"
            new_lines.append(new_line)
            fixed_count += 1
        else:
            new_lines.append(line)

    # Overwrite the file if we made fixes
    if fixed_count > 0:
        with open(label_path, 'w') as f:
            f.writelines(new_lines)

    return fixed_count

def scan_and_fix():
    print(f"Scanning dataset at: {DATASET_DIR}")
    total_fixed = 0

    # Check train, valid, and test label folders
    for split in ['train', 'valid', 'test', 'val']:
        labels_dir = DATASET_DIR / split / 'labels'

        if labels_dir.exists():
            print(f"Checking {split}/labels...")
            for label_file in labels_dir.glob('*.txt'):
                fixed = fix_bbox_to_polygon(label_file)
                if fixed > 0:
                    print(f"  Fixed {fixed} bounding boxes in {label_file.name}")
                    total_fixed += fixed

    print(f"\nDone! Fixed a total of {total_fixed} bounding boxes.")

if __name__ == "__main__":
    scan_and_fix()

In [ ]:
!rm /content/fire123_seg-1/train/labels.cache
!rm /content/fire123_seg-1/valid/labels.cache

## Save Metrics

In [ ]:
import shutil

source_folder = "/content/runs/segment/val"
destination_folder = "/content/drive/MyDrive/Models/firesmoke_4_seg-metrics"

try:
    shutil.copytree(source_folder, destination_folder)
    print("Successfully backed up training results and best.pt to Google Drive!")
except Exception as e:
    print(f"Error copying files: {e}")

In [ ]:
!rm -rf /content/output